In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
import re
import matplotlib
matplotlib.use('Agg')        
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils import resample

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
POS = '#1D9E75'   # green  = positive
NEU = '#EF9F27'   # amber  = neutral
NEG = '#D85A30'   # red    = negative
BLU = '#3B8BD4'   # blue   = comparison bars

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

In [3]:
df = pd.read_csv('cleaned_reviews.csv')
print(f"Loaded {len(df)} reviews")
print("Sentiment counts:", df['sentiment'].value_counts().to_dict())

Loaded 1162 reviews
Sentiment counts: {'positive': 964, 'neutral': 124, 'negative': 74}


In [4]:
pos = df[df['sentiment'] == 'positive']
neu = df[df['sentiment'] == 'neutral']
neg = df[df['sentiment'] == 'negative']

target        = len(pos) // 3
neu_upsampled = resample(neu, replace=True, n_samples=target, random_state=42)
neg_upsampled = resample(neg, replace=True, n_samples=target, random_state=42)
df_balanced   = pd.concat([pos, neu_upsampled, neg_upsampled])

print(f"After balancing: {df_balanced['sentiment'].value_counts().to_dict()}")


After balancing: {'positive': 964, 'neutral': 321, 'negative': 321}


In [5]:
X = df_balanced['clean_text']
y = df_balanced['sentiment']

tfidf        = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_vectorized = tfidf.fit_transform(X)

print(f"TF-IDF shape: {X_vectorized.shape[0]} reviews × {X_vectorized.shape[1]} features")


TF-IDF shape: 1606 reviews × 5000 features


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}")


Training: 1284 | Testing: 322


In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes':         MultinomialNB(alpha=0.5),
    'Linear SVM':          LinearSVC(max_iter=2000, random_state=42),
}

print("-" * 55)
print("MODEL COMPARISON")
print("-" * 55)

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    cv_acc   = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()
    results[name] = {'model': model, 'accuracy': test_acc, 'cv': cv_acc}
    print(f"{name:25s}  Test: {test_acc:.1%}   CV: {cv_acc:.1%}")

best_name  = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
print(f"Best model: {best_name} ({results[best_name]['accuracy']:.1%} accuracy)")

-------------------------------------------------------
MODEL COMPARISON
-------------------------------------------------------
Logistic Regression        Test: 94.1%   CV: 90.0%
Naive Bayes                Test: 88.2%   CV: 85.4%
Linear SVM                 Test: 97.5%   CV: 96.4%
Best model: Linear SVM (97.5% accuracy)


In [9]:
y_pred = best_model.predict(X_test)

print("=" * 55)
print("CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred))

cm_data = confusion_matrix(y_test, y_pred, labels=['positive', 'neutral', 'negative'])
print("Confusion Matrix:")
print(pd.DataFrame(cm_data,
    index  =['Actual: positive', 'Actual: neutral', 'Actual: negative'],
    columns=['Pred: positive',   'Pred: neutral',   'Pred: negative']))


CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       0.96      0.98      0.97        65
     neutral       0.97      0.97      0.97        64
    positive       0.98      0.97      0.98       193

    accuracy                           0.98       322
   macro avg       0.97      0.98      0.97       322
weighted avg       0.98      0.98      0.98       322

Confusion Matrix:
                  Pred: positive  Pred: neutral  Pred: negative
Actual: positive             188              2               3
Actual: neutral                2             62               0
Actual: negative               1              0              64


In [10]:
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("model.pkl saved!")
print("tfidf.pkl saved!")

model.pkl saved!
tfidf.pkl saved!


In [11]:
stop_words = set(stopwords.words('english'))

def clean_new_review(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'<.*?>', '', text)              # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)          # letters only
    text = re.sub(r'\s+', ' ', text).strip()       # fix spaces
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

# Quick live test
test_reviews = [
    "This kindle is absolutely amazing. Love reading on it every night!",
    "Battery dies too fast. Very disappointed with this product.",
    "It is okay. Nothing special but works fine for basic use.",
    "Best purchase I ever made. Perfect gift for my daughter!",
    "Stopped working after 2 weeks. Total waste of money.",
]

print("=" * 55)
print("LIVE PREDICTIONS")
print("=" * 55)
for review in test_reviews:
    cleaned = clean_new_review(review)
    pred    = best_model.predict(tfidf.transform([cleaned]))[0]
    icon    = '😊' if pred=='positive' else ('😐' if pred=='neutral' else '😠')
    print(f"{icon} {pred.upper():10s} ← {review[:55]}...")

LIVE PREDICTIONS
😊 POSITIVE   ← This kindle is absolutely amazing. Love reading on it e...
😊 POSITIVE   ← Battery dies too fast. Very disappointed with this prod...
😠 NEGATIVE   ← It is okay. Nothing special but works fine for basic us...
😊 POSITIVE   ← Best purchase I ever made. Perfect gift for my daughter...
😠 NEGATIVE   ← Stopped working after 2 weeks. Total waste of money....


Model Comparsion

In [12]:
names    = list(results.keys())
test_acc = [results[n]['accuracy'] * 100 for n in names]
cv_acc   = [results[n]['cv']       * 100 for n in names]
x = np.arange(len(names))
w = 0.32

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - w/2, test_acc, w, color=BLU, label='Test Accuracy', edgecolor='white')
bars2 = ax.bar(x + w/2, cv_acc,   w, color=POS, label='CV Accuracy',   edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Highlight winner
winner_idx = names.index(best_name)
ax.axvspan(winner_idx - 0.4, winner_idx + 0.4, alpha=0.08, color=POS, zorder=0)
ax.text(winner_idx, 103, 'BEST', ha='center', fontsize=10, color=POS, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(75, 106)
ax.set_ylabel('Accuracy %', fontsize=11)
ax.set_title('Model Comparison — Which model is best?', fontsize=13, fontweight='bold')
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig('chartA_model_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartA_model_compare.png saved!")

chartA_model_compare.png saved!


Confusion Matrix

In [13]:
labels_cm = ['positive', 'neutral', 'negative']

fig, ax = plt.subplots(figsize=(5.5, 4.5))

for i in range(3):
    for j in range(3):
        val = cm_data[i][j]
        bg  = POS if i == j else ('#F0997B' if val > 5 else '#FAECE7')
        ax.add_patch(plt.Rectangle((j - 0.5, 2 - i - 0.5), 1, 1, color=bg, zorder=1))
        tc  = 'white' if i == j else ('#993C1D' if val > 5 else '#4A1B0C')
        ax.text(j, 2 - i, str(val),
                ha='center', va='center',
                fontsize=16, fontweight='bold', color=tc, zorder=2)

ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5)
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(labels_cm, fontsize=10)
ax.set_yticklabels(labels_cm[::-1], fontsize=10)
ax.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax.set_ylabel('Actual',    fontsize=11, fontweight='bold')
ax.set_title('Confusion Matrix\n(green = correct predictions)', fontsize=12, fontweight='bold')
ax.spines[:].set_visible(False)
ax.tick_params(length=0)

correct = mpatches.Patch(color=POS, label='Correct')
wrong   = mpatches.Patch(color='#F0997B', label='Wrong')
ax.legend(handles=[correct, wrong], loc='upper right', frameon=False,
          fontsize=9, bbox_to_anchor=(1.02, -0.12), ncol=2)

plt.tight_layout()
plt.savefig('chartB_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartB_confusion.png saved!")


chartB_confusion.png saved!


TF-IDF top wordsper sentiment

In [14]:
feature_names = tfidf.get_feature_names_out()
class_labels  = best_model.classes_
cc = {'positive': POS, 'neutral': NEU, 'negative': NEG}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for idx, (ax, cls) in enumerate(zip(axes, class_labels)):
    coefs   = best_model.coef_[idx]
    top_idx = np.argsort(coefs)[-10:][::-1]
    words   = [feature_names[i] for i in top_idx]
    scores  = [coefs[i]         for i in top_idx]

    ax.barh(words[::-1], scores[::-1], color=cc.get(cls, BLU), edgecolor='white')
    ax.set_title(f'{cls.title()} reviews', fontsize=11,
                 fontweight='bold', color=cc.get(cls, BLU))
    ax.set_xlabel('TF-IDF weight', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Top words the model uses to decide each sentiment',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chartC_tfidf_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartC_tfidf_features.png saved!")

chartC_tfidf_features.png saved!


Live Prediction Visual

In [15]:
preds_demo = [
    best_model.predict(tfidf.transform([clean_new_review(r)]))[0]
    for r in test_reviews
]

sc = {'positive': POS, 'neutral': NEU, 'negative': NEG}
sl = {'positive': 'POSITIVE', 'neutral': 'NEUTRAL', 'negative': 'NEGATIVE'}

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axis('off')
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.text(5, 9.5, 'Live Prediction Demo — Model in Action!',
        ha='center', fontsize=13, fontweight='bold', color='#2C2C2A')

for i, (rev, pred) in enumerate(zip(test_reviews, preds_demo)):
    y_pos = 8.2 - i * 1.55
    color = sc[pred]
    rect  = mpatches.FancyBboxPatch(
                (0.2, y_pos - 0.55), 9.55, 1.1,
                boxstyle='round,pad=0.08',
                facecolor=color + '25', edgecolor=color, linewidth=1.3)
    ax.add_patch(rect)
    short = (rev[:63] + '…') if len(rev) > 66 else rev
    ax.text(0.55, y_pos + 0.15, short, fontsize=9, va='center', color='#2C2C2A')
    ax.text(9.6,  y_pos - 0.12, sl[pred], fontsize=8.5, va='center',
            ha='right', color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('chartD_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartD_predictions.png saved!")

chartD_predictions.png saved!
